In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
"""
Preprocesamiento para el método de Lin & Nguyen (2020) - hybrid-sampling.
Extrae pools de puntos por clase desde el volumen denso y los guarda en .pt.
Clases: 0=bg, 1=riñón (incluye quiste), 2=tumor. HU: [-79, 304] -> [0, 1].
"""

import os, numpy as np, nibabel as nib, torch
from scipy.ndimage import zoom

# --- Configuración ---
BASE           = '/content/drive/MyDrive/kits2'
OUT_DIR        = f'{BASE}/pools_hybrid'
HU_MIN, HU_MAX = -79, 304
OUT_SHAPE      = (128, 128, 128)
CAP_BG         = 200_000     # Tope de puntos de fondo a guardar
CAP_K          = 60_000      # Tope de puntos de riñón
CAP_T          = 40_000      # Tope de puntos de tumor
os.makedirs(OUT_DIR, exist_ok=True)

# --- NIfTI a Volumen Denso ---
def nifti_to_dense(case):
    img_nii = nib.as_closest_canonical(nib.load(f'{BASE}/{case}/imaging.nii.gz'))
    seg_nii = nib.as_closest_canonical(nib.load(f'{BASE}/{case}/segmentation.nii.gz'))
    img = img_nii.get_fdata().astype(np.float32)
    seg = seg_nii.get_fdata().astype(np.uint8)

    lab = np.zeros_like(seg, dtype=np.int64)
    lab[seg==1] = 1; lab[seg==2] = 2; lab[seg==3] = 1
    if lab.max() == 0: return None, None

    img = np.clip(img, HU_MIN, HU_MAX); img = (img - HU_MIN) / (HU_MAX - HU_MIN)
    zf = [t/s for t,s in zip(OUT_SHAPE, img.shape)]
    img = zoom(img, zf, order=1).astype(np.float32)
    lab = zoom(lab, zf, order=0).astype(np.int64)

    return img, lab

# --- Extracción de Pools por Clase ---
def extract_pool(vol, lab, cls, cap):
    D, H, W = vol.shape
    zz, yy, xx = np.where(lab == cls)
    n = len(zz)

    if n == 0:
        return np.zeros((0,3), np.float32), np.zeros((0,1), np.float32)

    if n > cap:
        keep = np.random.choice(n, cap, replace=False)
        zz, yy, xx = zz[keep], yy[keep], xx[keep]

    coords = np.stack([xx/max(W-1,1), yy/max(H-1,1), zz/max(D-1,1)], 1).astype(np.float32)
    hu = vol[zz, yy, xx].astype(np.float32)[:, None]

    return coords, hu

# --- Procesamiento por Caso ---
def process_case(case):
    out_p = f'{OUT_DIR}/{case}.pt'

    if os.path.exists(out_p): return 'ya', 0
    if not os.path.exists(f'{BASE}/{case}/imaging.nii.gz'): return 'falta', 0

    vol, lab = nifti_to_dense(case)
    if vol is None: return 'sin_label', 0

    cb, hb = extract_pool(vol, lab, 0, CAP_BG)
    ck, hk = extract_pool(vol, lab, 1, CAP_K)
    ct, ht = extract_pool(vol, lab, 2, CAP_T)

    torch.save({
        'coords_bg': torch.from_numpy(cb), 'hu_bg': torch.from_numpy(hb),
        'coords_k': torch.from_numpy(ck),  'hu_k': torch.from_numpy(hk),
        'coords_t': torch.from_numpy(ct),  'hu_t': torch.from_numpy(ht)
    }, out_p)

    return 'ok', len(ct)

# --- Flujo Principal ---
def main():
    cases = sorted(d for d in os.listdir(BASE) if d.startswith('case_'))
    print(f'Total casos: {len(cases)}')

    n = {'ok': 0, 'ya': 0, 'falta': 0, 'sin_label': 0}
    tcounts = []

    for i, c in enumerate(cases):
        r, nt = process_case(c)
        n[r] += 1

        if r == 'ok':
            tcounts.append(nt)
            if i % 10 == 0: print(f'  {i}/{len(cases)} ok {c} | puntos tumor frescos: {nt}')

    if tcounts:
        tc = np.array(tcounts)
        print(f'\nTumor (puntos densos frescos): min={tc.min()} max={tc.max()} media={int(tc.mean())}')
        print('Compara: la nube Canny+FPS solo tenia ~184 de tumor. Aqui hay miles.')

    print(f"Listo. nuevos:{n['ok']} | ya:{n['ya']} | sin archivos:{n['falta']} | sin label:{n['sin_label']}")

if __name__ == '__main__':
    main()

In [ ]:
"""
Pipeline de Entrenamiento y Evaluación: PointVoxelFormer + Hybrid Sampling
Implementa Curriculum Learning (STAGES) y Balanced Focus Loss.
"""

import os, math, time, json, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.spatial import cKDTree

# --- Configuración ---
DRIVE_BASE  = '/content/drive/MyDrive/kits2'
LOCAL_BASE  = '/content/kits_local'
POOLS_DIR   = f'{LOCAL_BASE}/pools_hybrid'
CKPT_DIR    = f'{DRIVE_BASE}/checkpoints_hybrid'

NUM_CLASSES = 3; C_CH = 64; N_BLOCKS = 4; GRID = 32; MLP_RATIO = 4
BATCH_SIZE  = 4; NUM_POINTS = 16384; LR = 1e-3
KNN_K       = 3; N_ENSEMBLE = 3; CHUNK = 300_000
VOL_SHAPE   = (128, 128, 128)
GAMMA       = 2.0
SEED        = 42

# Currículum (Época fin, {clase: proporción})
STAGES = [
    (50,  {0: 0.20, 1: 0.40, 2: 0.40}),
    (100, {0: 0.50, 1: 0.30, 2: 0.20}),
    (155, {0: 0.85, 1: 0.12, 2: 0.03}),
]

np.random.seed(SEED); torch.manual_seed(SEED)

# --- Utilidades y Datos ---
def stage_ratio(ep):
    for fin, r in STAGES:
        if ep < fin: return r
    return STAGES[-1][1]

def copy_to_local():
    import shutil
    os.makedirs(POOLS_DIR, exist_ok=True)
    src = f'{DRIVE_BASE}/pools_hybrid'
    if os.path.isdir(src):
        faltan = [f for f in os.listdir(src) if not os.path.exists(f'{POOLS_DIR}/{f}')]
        for i, f in enumerate(faltan):
            shutil.copy(f'{src}/{f}', f'{POOLS_DIR}/{f}')
            if i % 50 == 0: print(f'  pools: {i}/{len(faltan)}')
        print(f'pools_hybrid: {len(os.listdir(POOLS_DIR))} en local')

def sample_from_pool(d, ratio, n_points):
    pools = {0: (d['coords_bg'].numpy(), d['hu_bg'].numpy()),
             1: (d['coords_k'].numpy(),  d['hu_k'].numpy()),
             2: (d['coords_t'].numpy(),  d['hu_t'].numpy())}
    cs, hs, ls = [], [], []

    for c in (0, 1, 2):
        coords, hu = pools[c]; want = int(round(n_points * ratio[c]))
        if want <= 0 or len(coords) == 0: continue
        idx = np.random.choice(len(coords), want, replace=(len(coords) < want))
        cs.append(coords[idx]); hs.append(hu[idx]); ls.append(np.full(want, c, np.int64))

    coords = np.concatenate(cs); hu = np.concatenate(hs); labels = np.concatenate(ls)

    if len(coords) > n_points:
        k = np.random.choice(len(coords), n_points, replace=False)
        coords, hu, labels = coords[k], hu[k], labels[k]
    elif len(coords) < n_points:
        k = np.random.choice(len(coords), n_points - len(coords), replace=True)
        coords = np.concatenate([coords, coords[k]])
        hu = np.concatenate([hu, hu[k]])
        labels = np.concatenate([labels, labels[k]])

    return coords, hu, labels

class PoolDataset(Dataset):
    def __init__(self, files, ratio_fn, augment=False):
        self.files = files; self.ratio_fn = ratio_fn; self.augment = augment; self.ep = 0

    def set_epoch(self, ep):
        self.ep = ep

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        d = torch.load(self.files[i], weights_only=False)
        coords, hu, labels = sample_from_pool(d, self.ratio_fn(self.ep), NUM_POINTS)
        c = coords - coords.mean(0, keepdims=True)
        c = c / (np.max(np.linalg.norm(c, axis=1)) + 1e-6)

        if self.augment:
            a = np.random.uniform(0, 2 * math.pi)
            R = np.array([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0], [0, 0, 1]], np.float32)
            c = c @ R.T
            c += np.random.normal(0, 0.002, c.shape).astype(np.float32)
            c *= np.random.uniform(0.9, 1.1)

        return torch.from_numpy(c).float(), torch.from_numpy(hu).float(), torch.from_numpy(labels).long()

# --- Modelo PointVoxelFormer ---
def normalize_to_grid(xyz, grid):
    mn = xyz.amin(1, keepdim=True); mx = xyz.amax(1, keepdim=True)
    return (xyz - mn) / (mx - mn + 1e-6) * (grid - 1)

def trilinear_splat(xyz, feat, grid):
    B, N, Cc = feat.shape; dev = feat.device
    x0 = torch.floor(xyz).long().clamp(0, grid-2); xf = xyz - x0.float()
    vox = torch.zeros(B, Cc, grid, grid, grid, device=dev)
    wgt = torch.zeros(B, 1, grid, grid, grid, device=dev)

    for ddx in (0, 1):
        for ddy in (0, 1):
            for ddz in (0, 1):
                wx = xf[..., 0] if ddx else 1 - xf[..., 0]
                wy = xf[..., 1] if ddy else 1 - xf[..., 1]
                wz = xf[..., 2] if ddz else 1 - xf[..., 2]
                w = (wx * wy * wz).unsqueeze(-1)

                ix = (x0[..., 0] + ddx).clamp(0, grid-1)
                iy = (x0[..., 1] + ddy).clamp(0, grid-1)
                iz = (x0[..., 2] + ddz).clamp(0, grid-1)
                flat = (ix * grid * grid + iy * grid + iz)

                vox.view(B, Cc, -1).scatter_add_(2, flat.unsqueeze(1).expand(-1, Cc, -1), (w * feat).permute(0, 2, 1))
                wgt.view(B, 1, -1).scatter_add_(2, flat.unsqueeze(1), w.permute(0, 2, 1))
    return vox / (wgt + 1e-5)

def trilinear_sample(vox, xyz, grid):
    B, N = vox.shape[0], xyz.shape[1]
    g = (xyz / (grid - 1)) * 2 - 1
    g = g.view(B, N, 1, 1, 3)[..., [2, 1, 0]]
    return F.grid_sample(vox, g, mode='bilinear', align_corners=True).view(B, vox.shape[1], N).permute(0, 2, 1)

class PVFBlock(nn.Module):
    def __init__(self, Cc, grid, ratio=4):
        super().__init__(); self.grid = grid
        self.norm1 = nn.LayerNorm(Cc)
        self.mlp = nn.Sequential(nn.Linear(Cc, Cc * ratio), nn.GELU(), nn.Linear(Cc * ratio, Cc))
        self.norm2 = nn.LayerNorm(Cc)
        self.conv = nn.Sequential(
            nn.Conv3d(Cc, Cc, 3, padding=1, bias=False), nn.BatchNorm3d(Cc), nn.GELU(),
            nn.Conv3d(Cc, Cc, 3, padding=1, bias=False), nn.BatchNorm3d(Cc)
        )

    def forward(self, xg, feat):
        feat = feat + self.mlp(self.norm1(feat))
        return feat + trilinear_sample(self.conv(trilinear_splat(xg, self.norm2(feat), self.grid)), xg, self.grid)

class PointVoxelFormer(nn.Module):
    def __init__(self, in_feat=1, num_classes=3, Cc=64, grid=32, n_blocks=4, ratio=4):
        super().__init__(); self.grid = grid
        self.stem = nn.Sequential(nn.Linear(in_feat + 3, Cc), nn.GELU(), nn.Linear(Cc, Cc))
        self.blocks = nn.ModuleList([PVFBlock(Cc, grid, ratio) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(Cc), nn.Linear(Cc, Cc), nn.GELU(), nn.Dropout(0.3), nn.Linear(Cc, num_classes))

    def forward(self, xyz, feat):
        xg = normalize_to_grid(xyz, self.grid)
        x = self.stem(torch.cat([xyz, feat], -1))
        for blk in self.blocks: x = blk(xg, x)
        return self.head(x).transpose(1, 2)

# --- Pérdida y Evaluación ---
def balanced_focus_loss(logits, target, gamma=GAMMA):
    B, C, N = logits.shape
    logp = F.log_softmax(logits, 1); p = logp.exp()

    with torch.no_grad():
        counts = torch.bincount(target.view(-1), minlength=C).float()
        freq = counts / (counts.sum() + 1e-8)
        alpha = 1.0 / (freq + 1e-6)
        alpha = alpha / alpha.sum() * C

    tgt = target.unsqueeze(1)
    logp_t = logp.gather(1, tgt).squeeze(1); p_t = p.gather(1, tgt).squeeze(1)
    a_t = alpha[target]
    loss = -a_t * ((1 - p_t)**gamma) * logp_t

    return loss.mean()

def dice_pc(pred, gt, nc=NUM_CLASSES):
    out = []
    for c in range(nc):
        p = (pred == c).float(); g = (gt == c).float()
        i = (p * g).sum(); u = p.sum() + g.sum()
        out.append((2 * i / (u + 1e-8)).item() if u > 0 else float('nan'))
    return out

@torch.no_grad()
def evaluate_volume(model, coords, hu, device):
    cn = coords - coords.mean(0, keepdims=True)
    cn = cn / (np.max(np.linalg.norm(cn, 1)) + 1e-6)
    logits = np.zeros((len(coords), NUM_CLASSES), np.float32)

    for _ in range(N_ENSEMBLE):
        logits += model(
            torch.from_numpy(cn).float().unsqueeze(0).to(device),
            torch.from_numpy(hu).float().unsqueeze(0).to(device)
        ).squeeze(0).cpu().numpy().T

    pred_pts = logits.argmax(1)
    D, H, W = VOL_SHAPE
    dz = np.linspace(0, 1, D, dtype=np.float32); dy = np.linspace(0, 1, H, dtype=np.float32); dx = np.linspace(0, 1, W, dtype=np.float32)
    zz, yy, xx = np.meshgrid(dz, dy, dx, indexing='ij')
    vox = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], 1)

    tree = cKDTree(coords); pred_full = np.zeros(D * H * W, np.int64)
    for s in range(0, len(vox), CHUNK):
        e = min(s + CHUNK, len(vox))
        _, ki = tree.query(vox[s:e], k=KNN_K, workers=-1)
        v = pred_pts[ki]
        pred_full[s:e] = np.array([np.bincount(x, minlength=NUM_CLASSES).argmax() for x in v])

    return pred_full

def gt_volume(coords, labels):
    D, H, W = VOL_SHAPE
    dz = np.linspace(0, 1, D, dtype=np.float32); dy = np.linspace(0, 1, H, dtype=np.float32); dx = np.linspace(0, 1, W, dtype=np.float32)
    zz, yy, xx = np.meshgrid(dz, dy, dx, indexing='ij')
    vox = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], 1)
    _, ki = cKDTree(coords).query(vox, k=1, workers=-1)
    return labels[ki]

# --- Flujo Principal ---
def main():
    try:
        from google.colab import drive; drive.mount('/content/drive', force_remount=True)
    except Exception: pass

    os.makedirs(CKPT_DIR, exist_ok=True)
    print('Copiando pools a local...'); copy_to_local()
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', dev)

    files = sorted(os.path.join(POOLS_DIR, f) for f in os.listdir(POOLS_DIR) if f.endswith('.pt'))
    np.random.seed(SEED); np.random.shuffle(files); n = len(files)
    tr, va, te = files[:int(n*.8)], files[int(n*.8):int(n*.9)], files[int(n*.9):]
    print(f'train {len(tr)} | val {len(va)} | test {len(te)}')

    train_ds = PoolDataset(tr, stage_ratio, augment=True)
    train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
    val_ds = PoolDataset(va, lambda ep: STAGES[-1][1], augment=False)
    val_dl = DataLoader(val_ds, 1, shuffle=False, num_workers=2)

    model = PointVoxelFormer(1, NUM_CLASSES, C_CH, GRID, N_BLOCKS, MLP_RATIO).to(dev)
    print(f'parametros: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    NUM_EPOCHS = STAGES[-1][0]
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, NUM_EPOCHS, eta_min=1e-5)

    start, best = 0, 0.0
    last_ck, best_ck = f'{CKPT_DIR}/last.pth', f'{CKPT_DIR}/best.pth'

    if os.path.exists(last_ck):
        ck = torch.load(last_ck, weights_only=False, map_location=dev)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); start = ck['epoch'] + 1; best = ck.get('best', 0.0)
        print(f'reanudando epoch {start}')

    for ep in range(start, NUM_EPOCHS):
        t0 = time.time(); model.train(); tl = 0; train_ds.set_epoch(ep)
        r = stage_ratio(ep)

        for xyz, feat, lbl in train_dl:
            xyz, feat, lbl = xyz.to(dev), feat.to(dev), lbl.to(dev)
            opt.zero_grad()
            loss = balanced_focus_loss(model(xyz, feat), lbl)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item()

        tl /= len(train_dl); sched.step()
        model.eval(); vd = []

        with torch.no_grad():
            for xyz, feat, lbl in val_dl:
                xyz, feat, lbl = xyz.to(dev), feat.to(dev), lbl.to(dev)
                vd.append(dice_pc(model(xyz, feat).argmax(1).view(-1), lbl.view(-1)))

        vd = np.array(vd); K, T = np.nanmean(vd[:, 1]), np.nanmean(vd[:, 2]); fg = (K + T) / 2
        print(f'E{ep+1:03d} TL:{tl:.4f} K:{K:.4f} T:{T:.4f} FG:{fg:.4f} ratio_t={r[2]:.2f} ({time.time()-t0:.0f}s)')
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'best': best}, last_ck)

        if fg > best:
            best = fg
            torch.save({'epoch': ep, 'model': model.state_dict(), 'val_fg': fg, 'val_T': T}, best_ck)
            print(f'  best FG={fg:.4f} T={T:.4f}')

    # --- Test a volumen completo ---
    ck = torch.load(best_ck, weights_only=False, map_location=dev)
    model.load_state_dict(ck['model']); model.eval()
    print(f'\nTEST (best epoch {ck["epoch"]+1})')
    res = []

    for f in te:
        d = torch.load(f, weights_only=False)
        coords, hu, labels = sample_from_pool(d, STAGES[-1][1], NUM_POINTS)
        pred_full = evaluate_volume(model, coords, hu, dev)
        gt_full = gt_volume(coords, labels)
        dv = {}

        for c, nm in enumerate(['bg', 'kidney', 'tumor']):
            p = (pred_full == c).astype(float); g = (gt_full == c).astype(float)
            i = (p * g).sum(); u = p.sum() + g.sum()
            dv[nm] = float('nan') if u == 0 else float(2 * i / (u + 1e-8))

        res.append(dv)
        print(f'  {os.path.basename(f)}: K={dv["kidney"]:.3f} T={dv["tumor"]:.3f}')

    def m(k): return np.nanmean([r[k] for r in res]), np.nanstd([r[k] for r in res])
    print('\n=== VOLUMEN COMPLETO (Lin & Nguyen) ===')
    for nm in ['kidney', 'tumor']:
        mu, sd = m(nm); print(f'{nm:<8} {mu:.4f} ± {sd:.4f}')

    print('(baseline Canny+FPS: tumor 0.089 | Nowakowski 0.693)')
    json.dump({k: list(m(k)) for k in ['bg', 'kidney', 'tumor']}, open(f'{CKPT_DIR}/results.json', 'w'), indent=2)

if __name__ == '__main__':
    main()